# 5 · Cross-dataset transfer evaluation (MANY datasets, one notebook)

Everything — the 3 heads **and** the fusion — was trained on **TriviaQA** (notebooks 1–2). Here we generate *fresh* answers for **several** datasets, score them with those already-trained heads, fuse, and report the full metric set per dataset. **Nothing is re-fit on the targets**, so every row is held out. The headline question: does **FUSED stay on top even when the best *single* detector changes** between datasets?

**Datasets** (set in the config cell): `nq_open` + `squad` are unseen by the heads (BLEURT label valid → real transfer); `triviaqa` is the training set, scored **held-out** via a high `OFFSET` (3000, past the training range 1000–2200). **TruthfulQA is excluded here** (BLEURT label invalid) — use notebook 6's comparative NLI judge for it.

**This is a GPU pass** (Instruct fp16; ~2 model loads per dataset). You'll see a `generate+cache` tqdm bar, a `scoring BLEURT…` line, then a `TSV scoring` bar, per dataset — if a bar advances it's alive.

Each dataset is also saved to `data/<ds>_cross_eval.parquet` so you can re-plot without re-running.

In [ ]:
import os, sys
os.environ.setdefault('HF_HOME', r'D:/LLAMA CACHE/huggingface')  # reuse local LLaMA cache
sys.path.insert(0, os.path.abspath(os.path.join('..', 'hallking')))
import warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join('..','tools'))
from cross_eval import evaluate_many
# ---- CONFIG -------------------------------------------------------------------------------
DATASETS = ['nq_open', 'squad', 'triviaqa']   # truthfulqa -> notebook 6 (broken BLEURT here)
TRAIN    = 'triviaqa'                          # heads + fusion were trained on this
N        = 300                                 # questions per dataset
OFFSETS  = {'triviaqa': 3000}                  # keep TriviaQA held-out (training used 1000-2200)
# ------------------------------------------------------------------------------------------
SCORED = evaluate_many(DATASETS, train_ds=TRAIN, n=N, offsets=OFFSETS)  # live tqdm per dataset
print('done:', list(SCORED.keys()))

### Per-dataset metrics (AUROC / AUPR / Accuracy / Precision / Recall / F1)

In [ ]:
import os, sys
os.environ.setdefault('HF_HOME', r'D:/LLAMA CACHE/huggingface')  # reuse local LLaMA cache
sys.path.insert(0, os.path.abspath(os.path.join('..', 'hallking')))
import warnings; warnings.filterwarnings('ignore')
import metrics as M, pandas as pd, numpy as np
DETS = {'SEP':'sep_entropy', 'HalluShift':'hallushift', 'TSV':'tsv_margin', 'FUSED':'fused'}
METRICS = {}
for ds, (_, df) in SCORED.items():
    y = df['hallucination'].to_numpy()
    res = {}
    for name, col in DETS.items():
        s = df[col].to_numpy()
        m = M.detector_metrics(y, s, threshold=M.best_threshold(y, s))
        M.attach_curves(m, y, s)
        res[name] = m
    METRICS[ds] = res
    print(f'\n=== {ds}  (n={len(df)}, halluc={y.mean()*100:.1f}%) ===')
    print(M.summary_table(res).to_string())

### ROC / PR curves + confusion matrices per dataset

In [ ]:
import matplotlib.pyplot as plt
for ds, res in METRICS.items():
    fig, ax = plt.subplots(1, 2, figsize=(11,4))
    M.plot_roc(ax[0], res); M.plot_pr(ax[1], res)
    fig.suptitle(f'{ds} — ROC / PR'); plt.tight_layout(); plt.show()
    fig, axes = plt.subplots(1, 4, figsize=(15,3.4))
    for axx,(name,m) in zip(axes, res.items()):
        M.plot_confusion(axx, m['confusion_matrix'], title=f"{ds}:{name}\nF1={m['F1']:.2f}")
    plt.tight_layout(); plt.show()

### Headline — does FUSED stay on top across datasets?

Pivot tables (rows = dataset, cols = detector). Watch the **best single detector change** between rows while **FUSED stays ≥ the best single** — that is the robustness story.

In [ ]:
for metric in ['AUROC','AUPR','F1']:
    piv = pd.DataFrame({ds:{name:res[name][metric] for name in DETS}
                        for ds,res in METRICS.items()}).T.round(3)
    piv['best_single'] = piv[['SEP','HalluShift','TSV']].idxmax(axis=1)
    piv['FUSED_wins'] = piv['FUSED'] >= piv[['SEP','HalluShift','TSV']].max(axis=1)
    print(f'\n=== {metric} (rows=dataset, cols=detector) ===')
    print(piv.to_string())